<h3>문자, 재귀, 토큰 청킹</h3>
<p>하나의 짧은 배송·환불 규정을 세 가지 방법으로 나눕니다. 각 스플리터를 만든 직후 결과 본문과 길이를 확인합니다.</p>

In [1]:
# %pip은 이 노트북의 현재 커널에 필요한 패키지를 설치합니다.
# 버전을 고정하면 학생 환경에서도 같은 클래스와 인자를 사용할 수 있습니다.
%pip install -q "langchain-text-splitters==1.1.2" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# 세 실험에 같은 원문을 사용해야 분할 기준의 차이만 비교할 수 있습니다.
text = """배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.

환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다."""

print(text)

배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.

환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.


<h3>1단계: 문자 기준으로 나누기</h3>
<p><code>CharacterTextSplitter</code>는 지정한 구분자를 기준으로 조각을 만든 뒤, <code>chunk_size</code> 안에서 조각을 합칩니다.</p>

In [3]:
# 공백을 경계로 사용하고, 청크의 목표 길이를 35자로 정합니다. 나눌 경계가 없는 긴 조각은 이를 넘을 수 있습니다.
character_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=35,
    chunk_overlap=5,
)
character_chunks = character_splitter.split_text(text)

for index, chunk in enumerate(character_chunks, start=1):
    print(f"{index}: {chunk!r} | 문자 수={len(chunk)}")

1: '배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더' | 문자 수=31
2: '이틀이 더 걸립니다.\n\n환불은 수령 후 7일 이내 신청합니다.' | 문자 수=34
3: '사용한 상품은 환불할 수 없습니다.' | 문자 수=19


<h3>2단계: 재귀 방식으로 나누기</h3>
<p>재귀 스플리터는 빈 줄, 문장, 공백 순서로 경계를 찾습니다. 큰 경계를 보존할 수 없을 때 다음 경계를 사용합니다.</p>

In [4]:
# separators의 앞쪽 항목일수록 먼저 시도합니다.
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", ". ", " ", ""],
    chunk_size=35,
    chunk_overlap=5,
    keep_separator="end",
)
recursive_chunks = recursive_splitter.split_text(text)

for index, chunk in enumerate(recursive_chunks, start=1):
    print(f"{index}: {chunk!r} | 문자 수={len(chunk)}")

1: '배송은 결제 다음 날 시작합니다.' | 문자 수=18
2: '제주 지역은 이틀이 더 걸립니다.' | 문자 수=18
3: '환불은 수령 후 7일 이내 신청합니다.' | 문자 수=21
4: '사용한 상품은 환불할 수 없습니다.' | 문자 수=19


<h3>3단계: 조건 하나만 바꾸기</h3>
<p>재귀 방식은 그대로 두고 <code>chunk_size</code>만 35에서 50으로 바꿉니다. 청크 수와 본문이 어떻게 달라지는지 확인합니다.</p>

In [5]:
# 이전 셀과 비교할 때 chunk_size만 바꿉니다.
larger_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", ". ", " ", ""],
    chunk_size=50,
    chunk_overlap=5,
    keep_separator="end",
)
larger_chunks = larger_splitter.split_text(text)

print("35자 설정:", len(recursive_chunks), "개")
print("50자 설정:", len(larger_chunks), "개")
for chunk in larger_chunks:
    print(repr(chunk), "| 문자 수=", len(chunk))

35자 설정: 4 개
50자 설정: 2 개
'배송은 결제 다음 날 시작합니다. 제주 지역은 이틀이 더 걸립니다.' | 문자 수= 37
'환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.' | 문자 수= 41


<h3>4단계: 토큰 기준으로 나누기</h3>
<p>토큰 스플리터는 모델 입력 한도를 다룰 때 사용합니다. 한국어 한 글자와 한 토큰은 같은 단위가 아니므로 토큰 수를 직접 확인합니다.</p>

In [6]:
# from_tiktoken_encoder()는 토큰 수를 길이 함수로 사용하는 재귀 스플리터를 만듭니다.
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=20,
    chunk_overlap=4,
)
token_chunks = token_splitter.split_text(text)
encoding = tiktoken.get_encoding("cl100k_base")

for index, chunk in enumerate(token_chunks, start=1):
    token_count = len(encoding.encode(chunk))
    print(f"{index}: {chunk!r} | 토큰 수={token_count}")

1: '배송은 결제 다음 날 시작합니다. 제주 지역은' | 토큰 수=20
2: '지역은 이틀이 더 걸립니다.' | 토큰 수=18
3: '환불은 수령 후 7일 이내 신청합니다.' | 토큰 수=19
4: '사용한 상품은 환불할 수 없습니다.' | 토큰 수=17


<h3>일부러 실패해 보기</h3>
<p>오버랩을 청크 크기보다 크게 지정하면 스플리터 생성자가 설정 오류를 알립니다.</p>

In [7]:
try:
    # 잘못된 설정을 실제 생성자에 전달해 오류를 재현합니다.
    invalid_splitter = RecursiveCharacterTextSplitter(
        chunk_size=20,
        chunk_overlap=21,
    )
except ValueError as error:
    print(type(error).__name__, str(error))
else:
    raise AssertionError("청크 크기를 넘는 오버랩이 거부되어야 합니다.")


ValueError Got a larger chunk overlap (21) than chunk size (20), should be smaller.


<p><strong>수정 방법</strong> 오버랩을 청크 크기보다 작게 지정하고, 반환된 모든 청크의 길이를 확인합니다.</p>

In [8]:
# 오버랩 5자는 청크 크기 20자보다 작습니다. 수정한 설정으로 분할합니다.
valid_splitter = RecursiveCharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=5,
)
fixed_chunks = valid_splitter.split_text(text)

assert all(len(chunk) <= 20 for chunk in fixed_chunks)
print("청크 수:", len(fixed_chunks))
print("모든 청크가 20자 이하입니다:", True)

청크 수: 5
모든 청크가 20자 이하입니다: True
